In [2]:
from pathlib import Path

import polars as pl
import pandas as pd

from IPython.display import display

ROOT = Path.cwd().parent

DATASETS = {
    "trades": ROOT / "data/merged/910004",
    "depth": ROOT / "data/merged/920002",
    "combined": ROOT / "data/merged/920003",
}

for name, path in DATASETS.items():
    print(f"{name:>10} | exists={path.exists()} | {path}")

    trades | exists=True | /home/pasha/projects/bybit-history-downloader/data/merged/910004
     depth | exists=True | /home/pasha/projects/bybit-history-downloader/data/merged/920002
  combined | exists=True | /home/pasha/projects/bybit-history-downloader/data/merged/920003


In [3]:
import json
import pyarrow.parquet as pq

for name, path in DATASETS.items():
    manifest = json.loads((path / "manifest.json").read_text())

    files = sorted(path.rglob("*.parquet"))

    print(f"\n{'=' * 80}")
    print(f"MARKETFORGE — {name.upper()}")
    print(f"{'=' * 80}")

    print(f"Events : {manifest['events_written']:,}")
    print(f"Files  : {len(files)}")

    if files:
        print(f"\nFirst file: {files[0].relative_to(path)}")
        print("\nSchema:")
        print(pq.ParquetFile(files[0]).schema_arrow)


MARKETFORGE — TRADES
Events : 50,106,717
Files  : 16

First file: timeline/partition-000000/part-000000.parquet

Schema:
event_timestamp_ns: int64 not null
stream_rank: uint32 not null
event_kind: uint8 not null
dataset_id: uint64 not null
task_id: uint64 not null
event_ordinal: uint64 not null
canonical_row_offset: uint64 not null
canonical_row_count: uint64 not null
event_boundary: uint8
source_operation: uint8

MARKETFORGE — DEPTH
Events : 405,581,426
Files  : 64

First file: timeline/partition-000000/part-000000.parquet

Schema:
event_timestamp_ns: int64 not null
stream_rank: uint32 not null
dataset_id: uint64 not null
task_id: uint64 not null
event_ordinal: uint64 not null
canonical_row_offset: uint64 not null
canonical_row_count: uint64 not null
event_boundary: uint8 not null
source_operation: uint8 not null

MARKETFORGE — COMBINED
Events : 455,688,143
Files  : 64

First file: timeline/partition-000000/part-000000.parquet

Schema:
event_timestamp_ns: int64 not null
stream_rank: 

In [4]:
trades = pl.scan_parquet(
    str(DATASETS["trades"] / "**/*.parquet"),
    glob=True,
)

display(trades.head(20).collect())

event_timestamp_ns,stream_rank,event_kind,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,event_boundary,source_operation
i64,u32,u8,u64,u64,u64,u64,u64,u8,u8
1788220800000000000,3,0,93,2,62633,62633,1,null,null
1788220800000000000,3,0,93,2,62634,62634,1,null,null
1788220800000000000,3,0,93,2,62635,62635,1,null,null
1788220800000000000,3,0,93,2,62636,62636,1,null,null
1788220800000000000,3,0,93,2,62637,62637,1,null,null
…,…,…,…,…,…,…,…,…,…
1788220800000000000,3,0,93,2,62648,62648,1,null,null
1788220800000000000,3,0,93,2,62649,62649,1,null,null
1788220800000000000,3,0,93,2,62650,62650,1,null,null


In [5]:
depth = pl.scan_parquet(
    str(DATASETS["depth"] / "**/*.parquet"),
    glob=True,
)

display(depth.head(20).collect())

event_timestamp_ns,stream_rank,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,event_boundary,source_operation
i64,u32,u64,u64,u64,u64,u64,u8,u8
1788220800000000000,3,96,0,44061,0,44061,1,0
1788220800000000000,3,96,0,44062,44061,1,0,2
1788220800000000000,3,96,0,44063,44062,1,0,2
1788220800000000000,3,96,0,44064,44063,1,0,2
1788220800000000000,3,96,0,44065,44064,1,0,2
…,…,…,…,…,…,…,…,…
1788220800000000000,3,96,0,44076,44075,1,0,2
1788220800000000000,3,96,0,44077,44076,1,0,2
1788220800000000000,3,96,0,44078,44077,1,0,2


In [6]:
combined = pl.scan_parquet(
    str(DATASETS["combined"] / "**/*.parquet"),
    glob=True,
)

display(combined.head(30).collect())

event_timestamp_ns,stream_rank,event_kind,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,event_boundary,source_operation
i64,u32,u8,u64,u64,u64,u64,u64,u8,u8
1788220800000000000,3,0,93,2,62633,62633,1,null,null
1788220800000000000,3,0,93,2,62634,62634,1,null,null
1788220800000000000,3,0,93,2,62635,62635,1,null,null
1788220800000000000,3,0,93,2,62636,62636,1,null,null
1788220800000000000,3,0,93,2,62637,62637,1,null,null
…,…,…,…,…,…,…,…,…,…
1788220800000000000,12,0,109,0,47830,47830,1,null,null
1788220800000000000,12,0,109,0,47831,47831,1,null,null
1788220800000000000,12,0,109,0,47832,47832,1,null,null


In [7]:
preview = (
    combined
    .head(100)
    .with_columns(
        pl.from_epoch(
            pl.col("event_timestamp_ns"),
            time_unit="ns",
        ).alias("datetime_utc")
    )
    .collect()
)

display(preview)

event_timestamp_ns,stream_rank,event_kind,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,event_boundary,source_operation,datetime_utc
i64,u32,u8,u64,u64,u64,u64,u64,u8,u8,datetime[ns]
1788220800000000000,3,0,93,2,62633,62633,1,null,null,2026-09-01 00:00:00
1788220800000000000,3,0,93,2,62634,62634,1,null,null,2026-09-01 00:00:00
1788220800000000000,3,0,93,2,62635,62635,1,null,null,2026-09-01 00:00:00
1788220800000000000,3,0,93,2,62636,62636,1,null,null,2026-09-01 00:00:00
1788220800000000000,3,0,93,2,62637,62637,1,null,null,2026-09-01 00:00:00
…,…,…,…,…,…,…,…,…,…,…
1788220800000000000,24,1,96,0,44104,44103,1,0,2,2026-09-01 00:00:00
1788220800000000000,24,1,96,0,44105,44104,1,0,2,2026-09-01 00:00:00
1788220800000000000,24,1,96,0,44106,44105,1,0,2,2026-09-01 00:00:00


In [8]:
distribution = (
    combined
    .group_by("event_kind")
    .agg(pl.len().alias("events"))
    .collect()
    .sort("event_kind")
)

display(distribution)

event_kind,events
u8,u32
0,50106717
1,405581426


In [9]:
import json
from pathlib import Path

jobs_dir = ROOT / "data/.jobs/process"

metadata = []

for path in jobs_dir.glob("*.json"):
    job = json.loads(path.read_text())

    if not job.get("tasks"):
        continue

    task = job["tasks"][0]

    metadata.append({
        "dataset_id": job["dataset_id"],
        "exchange": task["exchange"],
        "symbol": task["symbol"],
        "stream_id": task["stream_id"],
        "content_type": job["output"]["content_type"],
    })

metadata = pl.DataFrame(metadata).unique(subset=["dataset_id"])

display(metadata.sort("dataset_id"))

dataset_id,exchange,symbol,stream_id,content_type
i64,str,str,str,str
88,"""bybit""","""BTCUSDT""","""bybit:perpetual:linear:BTCUSDT…","""trades"""
89,"""bybit""","""BTCUSDT""","""bybit:perpetual:linear:BTCUSDT…","""depth"""
90,"""binance""","""BTCUSDT""","""binance:perpetual:linear:BTCUS…","""trades"""
91,"""okx""","""BTC-USDT-SWAP""","""okx:perpetual:linear:BTC-USDT-…","""trades"""
92,"""okx""","""BTC-USDT-SWAP""","""okx:perpetual:linear:BTC-USDT-…","""depth"""
…,…,…,…,…
120,"""okx""","""BTC-USD_UM-261225""","""okx:future:linear:BTC-USD_UM-2…","""trades"""
121,"""okx""","""BTC-USD_UM-261225""","""okx:future:linear:BTC-USD_UM-2…","""depth"""
122,"""okx""","""BTC-USD-261225""","""okx:future:inverse:BTC-USD-261…","""trades"""


In [10]:
timeline = (
    combined
    .head(1000)
    .join(metadata.lazy(), on="dataset_id", how="left")
    .with_columns(
        pl.from_epoch(
            pl.col("event_timestamp_ns"),
            time_unit="ns",
        ).alias("datetime_utc"),

        pl.when(pl.col("event_kind") == 0)
        .then(pl.lit("TRADE"))
        .otherwise(pl.lit("DEPTH"))
        .alias("event_type"),
    )
    .select(
        "datetime_utc",
        "exchange",
        "symbol",
        "event_type",
        "dataset_id",
        "task_id",
        "event_ordinal",
        "canonical_row_offset",
        "canonical_row_count",
    )
    .collect()
)

display(timeline.head(50))

datetime_utc,exchange,symbol,event_type,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count
datetime[ns],str,str,str,u64,u64,u64,u64,u64
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",93,2,62633,62633,1
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",93,2,62634,62634,1
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",93,2,62635,62635,1
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",93,2,62636,62636,1
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",93,2,62637,62637,1
…,…,…,…,…,…,…,…,…
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",109,0,47850,47850,1
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",109,0,47851,47851,1
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""TRADE""",109,0,47852,47852,1


In [12]:
import json
from pathlib import Path

dataset_id = 93

# Locate the actual canonical dataset using its manifest.
matches = []

for path in (ROOT / "data").rglob("manifest.json"):
    if "merged" in path.parts:
        continue

    try:
        manifest = json.loads(path.read_text())
    except (OSError, json.JSONDecodeError):
        continue

    if manifest.get("dataset_id") == dataset_id:
        matches.append(path.parent)

assert len(matches) == 1, (
    f"Expected one dataset {dataset_id}, found: {matches}"
)

dataset_path = matches[0]
manifest = json.loads((dataset_path / "manifest.json").read_text())

print("DATASET:", dataset_id)
print("PATH:", dataset_path)

print("\nCANONICAL FILES:")

for entry in manifest["files"][:10]:
    print(entry["path"], "| rows:", entry.get("rows"))

DATASET: 93
PATH: /home/pasha/projects/bybit-history-downloader/data/.work/full-trade-test/datasets/bitget-perpetual-linear-BTCUSDT-trade-20260901-20260904-d93

CANONICAL FILES:
tasks/task-000000/trades/part-000000.parquet | rows: 100000
tasks/task-000001/trades/part-000000.parquet | rows: 100000
tasks/task-000002/trades/part-000000.parquet | rows: 100000
tasks/task-000003/trades/part-000000.parquet | rows: 100000
tasks/task-000004/trades/part-000000.parquet | rows: 100000
tasks/task-000005/trades/part-000000.parquet | rows: 100000
tasks/task-000006/trades/part-000000.parquet | rows: 100000
tasks/task-000007/trades/part-000000.parquet | rows: 100000
tasks/task-000008/trades/part-000000.parquet | rows: 100000
tasks/task-000009/trades/part-000000.parquet | rows: 83742


In [15]:
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow as pa
import polars as pl

from IPython.display import display

# Read only 30 rows from the first canonical trade file.
parquet = pq.ParquetFile(trade_files[0])

batch = next(parquet.iter_batches(batch_size=30))
table = pa.Table.from_batches([batch])

# Convert Decimal256 columns to Float64 for visualization.
columns = []

for field in table.schema:
    column = table[field.name]

    if pa.types.is_decimal256(field.type):
        column = pc.cast(column, pa.float64())

    columns.append(column)

table = pa.table(
    {
        field.name: column
        for field, column in zip(table.schema, columns)
    }
)

trades_preview = pl.from_arrow(table)

display(trades_preview)

event_timestamp_ns,system_timestamp_ns,exchange,instrument_id,symbol,stream_id,trade_id,sequence,side,price,quantity_base,quantity_quote,quantity_contracts,is_rpi,trade_iv,mark_iv,index_price,mark_price
i64,i64,str,i64,str,str,str,u64,str,f64,f64,f64,f64,bool,f64,f64,f64,f64
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349886673666048""",null,"""buy""",78546.0,0.0001,7.8546,null,null,null,null,null,null
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349886673666050""",null,"""buy""",78546.0,0.0001,7.8546,null,null,null,null,null,null
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349886673666052""",null,"""buy""",78546.0,0.0001,7.8546,null,null,null,null,null,null
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349886673666054""",null,"""buy""",78546.0,0.0001,7.8546,null,null,null,null,null,null
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349886673666056""",null,"""buy""",78546.0,0.0001,7.8546,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349886757552144""",null,"""buy""",78554.7,0.0006,47.13282,null,null,null,null,null,null
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349886757552146""",null,"""buy""",78554.9,0.0004,31.42196,null,null,null,null,null,null
1788192000000000000,null,"""bitget""",14215,"""BTCUSDT""","""bitget:perpetual:linear:BTCUSD…","""1478349887005016064""",null,"""buy""",78554.9,0.0077,604.87273,null,null,null,null,null,null


In [16]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import polars as pl

from IPython.display import display

depth_path = ROOT / "data/processed/96"

depth_files = sorted(
    depth_path.glob("tasks/task-*/depth/part-*.parquet")
)

assert depth_files, "No canonical depth files found"

print("MARKETFORGE — CANONICAL DEPTH")
print("=" * 80)
print("Files:", len(depth_files))
print("\nSchema:")
print(pq.ParquetFile(depth_files[0]).schema_arrow)

parquet = pq.ParquetFile(depth_files[0])
batch = next(parquet.iter_batches(batch_size=30))

table = pa.Table.from_batches([batch])

# Decimal256 -> Float64 for visualization only
table = pa.table({
    field.name: (
        pc.cast(table[field.name], pa.float64())
        if pa.types.is_decimal256(field.type)
        else table[field.name]
    )
    for field in table.schema
})

depth_preview = pl.from_arrow(table)

display(depth_preview)

MARKETFORGE — CANONICAL DEPTH
Files: 173

Schema:
event_timestamp_ns: int64 not null
system_timestamp_ns: int64
exchange: string not null
instrument_id: int64 not null
symbol: string not null
stream_id: string not null
side: string not null
price: decimal256(76, 28) not null
quantity_base: decimal256(76, 28)
quantity_quote: decimal256(76, 28)
quantity_contracts: decimal256(76, 28)
order_count: uint64


event_timestamp_ns,system_timestamp_ns,exchange,instrument_id,symbol,stream_id,side,price,quantity_base,quantity_quote,quantity_contracts,order_count
i64,i64,str,i64,str,str,str,f64,f64,f64,f64,u64
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",0.1,375.9051,37.59051,3.759051e6,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",0.2,13.4226,2.68452,134226.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",0.3,16.1094,4.83282,161094.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",0.4,1.0697,0.42788,10697.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",0.5,413.8663,206.93315,4.138663e6,null
…,…,…,…,…,…,…,…,…,…,…,…
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",5.4,0.0005,0.0027,5.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",5.9,0.0011,0.00649,11.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",6.0,0.0035,0.021,35.0,null


In [17]:
depth_view = depth_preview

if "event_timestamp_ns" in depth_view.columns:
    depth_view = depth_view.with_columns(
        pl.from_epoch(
            "event_timestamp_ns",
            time_unit="ns",
        ).alias("datetime_utc")
    )

preferred = [
    "datetime_utc",
    "exchange",
    "symbol",
    "side",
    "price",
    "quantity",
    "quantity_base",
    "operation",
    "sequence",
]

display(
    depth_view.select(
        [col for col in preferred if col in depth_view.columns]
    )
)

datetime_utc,exchange,symbol,side,price,quantity_base
datetime[ns],str,str,str,f64,f64
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",0.1,375.9051
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",0.2,13.4226
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",0.3,16.1094
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",0.4,1.0697
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",0.5,413.8663
…,…,…,…,…,…
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",5.4,0.0005
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",5.9,0.0011
2026-09-01 00:00:00,"""gateio""","""BTC_USDT""","""bid""",6.0,0.0035


In [18]:
def prepare_market_view(df, event_type):
    columns = df.columns

    timestamp = (
        "event_timestamp_ns"
        if "event_timestamp_ns" in columns
        else None
    )

    if timestamp is None:
        raise ValueError(f"No event timestamp in {event_type} schema")

    price = "price" if "price" in columns else None

    quantity = next(
        (
            name
            for name in ["quantity_base", "quantity", "size"]
            if name in columns
        ),
        None,
    )

    return (
        df
        .with_columns(
            pl.lit(event_type).alias("event_type"),
            pl.from_epoch(
                timestamp,
                time_unit="ns",
            ).alias("datetime_utc"),
        )
        .select(
            "datetime_utc",
            "event_timestamp_ns",
            "event_type",
            *[
                name
                for name in [
                    "exchange",
                    "symbol",
                    "side",
                ]
                if name in columns
            ],
            (
                pl.col(price).alias("price")
                if price
                else pl.lit(None).alias("price")
            ),
            (
                pl.col(quantity).alias("quantity")
                if quantity
                else pl.lit(None).alias("quantity")
            ),
        )
    )


trade_view = prepare_market_view(
    trades_preview,
    "TRADE",
)

book_view = prepare_market_view(
    depth_preview,
    "DEPTH",
)

market_activity = (
    pl.concat(
        [trade_view, book_view],
        how="diagonal_relaxed",
    )
    .sort("event_timestamp_ns", "event_type")
)

display(market_activity)

datetime_utc,event_timestamp_ns,event_type,exchange,symbol,side,price,quantity
datetime[ns],i64,str,str,str,str,f64,f64
2026-08-31 16:00:00,1788192000000000000,"""TRADE""","""bitget""","""BTCUSDT""","""buy""",78546.0,0.0001
2026-08-31 16:00:00,1788192000000000000,"""TRADE""","""bitget""","""BTCUSDT""","""buy""",78546.0,0.0001
2026-08-31 16:00:00,1788192000000000000,"""TRADE""","""bitget""","""BTCUSDT""","""buy""",78546.0,0.0001
2026-08-31 16:00:00,1788192000000000000,"""TRADE""","""bitget""","""BTCUSDT""","""buy""",78546.0,0.0001
2026-08-31 16:00:00,1788192000000000000,"""TRADE""","""bitget""","""BTCUSDT""","""buy""",78546.0,0.0001
…,…,…,…,…,…,…,…
2026-09-01 00:00:00,1788220800000000000,"""DEPTH""","""gateio""","""BTC_USDT""","""bid""",5.4,0.0005
2026-09-01 00:00:00,1788220800000000000,"""DEPTH""","""gateio""","""BTC_USDT""","""bid""",5.9,0.0011
2026-09-01 00:00:00,1788220800000000000,"""DEPTH""","""gateio""","""BTC_USDT""","""bid""",6.0,0.0035


In [19]:
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

from pathlib import Path
from IPython.display import display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "engine/Cargo.toml").exists()
)

MERGED = ROOT / "data/merged/920003"

# Read the merged timeline, not canonical data.
timeline_files = sorted(MERGED.rglob("*.parquet"))

print("Merged Parquet files:", len(timeline_files))

timeline = pl.scan_parquet(
    [str(p) for p in timeline_files]
)

# Take a bounded sample from the actual merged timeline.
sample = timeline.head(1000).collect()

print("Event distribution:")
display(sample.group_by("event_kind").len())

display(sample.head(20))

Merged Parquet files: 64
Event distribution:


event_kind,len
u8,u32
0,337
1,663


event_timestamp_ns,stream_rank,event_kind,dataset_id,task_id,event_ordinal,canonical_row_offset,canonical_row_count,event_boundary,source_operation
i64,u32,u8,u64,u64,u64,u64,u64,u8,u8
1788220800000000000,3,0,93,2,62633,62633,1,null,null
1788220800000000000,3,0,93,2,62634,62634,1,null,null
1788220800000000000,3,0,93,2,62635,62635,1,null,null
1788220800000000000,3,0,93,2,62636,62636,1,null,null
1788220800000000000,3,0,93,2,62637,62637,1,null,null
…,…,…,…,…,…,…,…,…,…
1788220800000000000,3,0,93,2,62648,62648,1,null,null
1788220800000000000,3,0,93,2,62649,62649,1,null,null
1788220800000000000,3,0,93,2,62650,62650,1,null,null


In [20]:
import json

dataset_paths = {}

for manifest_path in (ROOT / "data").rglob("manifest.json"):
    if "merged" in manifest_path.parts:
        continue

    try:
        manifest = json.loads(manifest_path.read_text())
    except (OSError, json.JSONDecodeError):
        continue

    dataset_id = manifest.get("dataset_id")

    if dataset_id is None:
        continue

    dataset_paths.setdefault(dataset_id, []).append(
        manifest_path.parent
    )

# Resolve only datasets referenced by our sample.
required_ids = sample["dataset_id"].unique().to_list()

for dataset_id in required_ids:
    matches = dataset_paths.get(dataset_id, [])

    if len(matches) != 1:
        raise RuntimeError(
            f"Dataset {dataset_id}: expected one location, found {matches}"
        )

print(f"Resolved {len(required_ids)} canonical datasets")

Resolved 19 canonical datasets


In [21]:
from functools import lru_cache

@lru_cache(maxsize=128)
def task_file_inventory(dataset_id, task_id, event_kind):
    dataset_path = dataset_paths[dataset_id][0]

    manifest = json.loads(
        (dataset_path / "manifest.json").read_text()
    )

    kind = "trades" if event_kind == 0 else "depth"

    prefix = f"tasks/task-{task_id:06d}/{kind}/"

    files = [
        entry
        for entry in manifest["files"]
        if entry["path"].startswith(prefix)
        and entry["path"].endswith(".parquet")
    ]

    files.sort(key=lambda entry: entry["path"])

    inventory = []
    offset = 0

    for entry in files:
        rows = entry["rows"]

        inventory.append({
            "path": dataset_path / entry["path"],
            "start": offset,
            "end": offset + rows,
        })

        offset += rows

    return inventory


def read_canonical_rows(
    dataset_id,
    task_id,
    event_kind,
    row_offset,
    row_count,
):
    if row_count == 0:
        return pa.table({})

    inventory = task_file_inventory(
        dataset_id,
        task_id,
        event_kind,
    )

    requested_start = row_offset
    requested_end = row_offset + row_count

    batches = []

    for entry in inventory:
        overlap_start = max(requested_start, entry["start"])
        overlap_end = min(requested_end, entry["end"])

        if overlap_start >= overlap_end:
            continue

        local_start = overlap_start - entry["start"]
        local_count = overlap_end - overlap_start

        parquet = pq.ParquetFile(entry["path"])

        # Read the relevant row groups.
        row_position = 0

        for row_group in range(parquet.num_row_groups):
            metadata = parquet.metadata.row_group(row_group)
            group_rows = metadata.num_rows

            group_start = row_position
            group_end = row_position + group_rows

            row_position = group_end

            if group_end <= local_start:
                continue

            if group_start >= local_start + local_count:
                break

            table = parquet.read_row_group(row_group)

            slice_start = max(local_start - group_start, 0)
            slice_end = min(
                local_start + local_count - group_start,
                group_rows,
            )

            batches.append(
                table.slice(
                    slice_start,
                    slice_end - slice_start,
                )
            )

    if not batches:
        raise RuntimeError(
            f"Canonical rows not found: "
            f"dataset={dataset_id}, task={task_id}, "
            f"offset={row_offset}, count={row_count}"
        )

    return pa.concat_tables(batches)

In [22]:
def arrow_to_polars(table):
    if table.num_rows == 0:
        return None

    columns = {}

    for field in table.schema:
        column = table[field.name]

        if pa.types.is_decimal256(field.type):
            column = pc.cast(column, pa.float64())

        columns[field.name] = column

    return pl.from_arrow(pa.table(columns))

In [23]:
trade_refs = (
    sample
    .filter(pl.col("event_kind") == 0)
    .head(10)
)

resolved_trades = []

for ref in trade_refs.iter_rows(named=True):
    table = read_canonical_rows(
        dataset_id=ref["dataset_id"],
        task_id=ref["task_id"],
        event_kind=0,
        row_offset=ref["canonical_row_offset"],
        row_count=ref["canonical_row_count"],
    )

    frame = arrow_to_polars(table)

    if frame is None:
        continue

    frame = frame.with_columns(
        pl.lit(ref["dataset_id"]).alias("dataset_id"),
        pl.lit(ref["task_id"]).alias("task_id"),
        pl.lit(ref["event_ordinal"]).alias("event_ordinal"),
    )

    resolved_trades.append(frame)

trades_real = pl.concat(
    resolved_trades,
    how="diagonal_relaxed",
)

display(
    trades_real
    .with_columns(
        pl.from_epoch(
            "event_timestamp_ns",
            time_unit="ns",
        ).alias("datetime_utc")
    )
    .select(
        "datetime_utc",
        "exchange",
        "symbol",
        "side",
        "price",
        "quantity_base",
        "quantity_quote",
        "trade_id",
        "dataset_id",
        "task_id",
        "event_ordinal",
    )
)

datetime_utc,exchange,symbol,side,price,quantity_base,quantity_quote,trade_id,dataset_id,task_id,event_ordinal
datetime[ns],str,str,str,f64,f64,f64,str,i32,i32,i32
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78552.8,0.002,157.1056,"""1478470684897984512""",93,2,62633
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78552.8,0.0006,47.13168,"""1478470684897984514""",93,2,62634
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78552.8,0.0045,353.4876,"""1478470685208363008""",93,2,62635
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78552.8,0.006,471.3168,"""1478470685220945920""",93,2,62636
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78552.8,0.0001,7.85528,"""1478470685220945922""",93,2,62637
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78551.2,0.0001,7.85512,"""1478470685220945924""",93,2,62638
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78550.4,0.0097,761.93888,"""1478470686437294080""",93,2,62639
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78550.4,0.0001,7.85504,"""1478470686437294082""",93,2,62640
2026-09-01 00:00:00,"""bitget""","""BTCUSDT""","""sell""",78550.4,0.0057,447.73728,"""1478470686437294084""",93,2,62641


In [24]:
depth_refs = (
    sample
    .filter(
        (pl.col("event_kind") == 1)
        & (pl.col("canonical_row_count") > 0)
        & (pl.col("canonical_row_count") <= 10)
    )
    .head(10)
)

resolved_depth = []

for ref in depth_refs.iter_rows(named=True):
    table = read_canonical_rows(
        dataset_id=ref["dataset_id"],
        task_id=ref["task_id"],
        event_kind=1,
        row_offset=ref["canonical_row_offset"],
        row_count=ref["canonical_row_count"],
    )

    frame = arrow_to_polars(table)

    if frame is None:
        continue

    frame = frame.with_columns(
        pl.lit(ref["dataset_id"]).alias("dataset_id"),
        pl.lit(ref["task_id"]).alias("task_id"),
        pl.lit(ref["event_ordinal"]).alias("event_ordinal"),
    )

    resolved_depth.append(frame)

if resolved_depth:
    depth_real = pl.concat(
        resolved_depth,
        how="diagonal_relaxed",
    )

    print("CANONICAL DEPTH SCHEMA")
    print(depth_real.schema)

    display(depth_real.head(30))
else:
    print("No suitable depth updates in the selected sample.")

CANONICAL DEPTH SCHEMA
Schema({'event_timestamp_ns': Int64, 'system_timestamp_ns': Int64, 'exchange': String, 'instrument_id': Int64, 'symbol': String, 'stream_id': String, 'side': String, 'price': Float64, 'quantity_base': Float64, 'quantity_quote': Float64, 'quantity_contracts': Float64, 'order_count': UInt64, 'dataset_id': Int32, 'task_id': Int32, 'event_ordinal': Int32})


event_timestamp_ns,system_timestamp_ns,exchange,instrument_id,symbol,stream_id,side,price,quantity_base,quantity_quote,quantity_contracts,order_count,dataset_id,task_id,event_ordinal
i64,i64,str,i64,str,str,str,f64,f64,f64,f64,u64,i32,i32,i32
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",60863.5,0.0009,54.77715,9.0,null,96,0,44062
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",78542.1,0.2153,16910.11413,2153.0,null,96,0,44063
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""ask""",79283.0,0.0008,63.4264,8.0,null,96,0,44064
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""ask""",79314.8,43.5359,3.4530e6,435359.0,null,96,0,44065
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",60876.1,0.0003,18.26283,3.0,null,96,0,44066
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",60863.5,0.0008,48.6908,8.0,null,96,0,44067
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",78542.5,0.2153,16910.20025,2153.0,null,96,0,44068
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",78464.2,null,null,0.0,null,96,0,44069
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_US…","""bid""",78462.8,19.1104,1.4995e6,191104.0,null,96,0,44070


In [26]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import polars as pl

# Read only the first 20 records
pf = pq.ParquetFile(files[0])
batch = next(pf.iter_batches(batch_size=20))
table = pa.Table.from_batches([batch])

# Convert Decimal256 to Float64 for display only
table_preview = pa.table({
    field.name: (
        pc.cast(table[field.name], pa.float64())
        if pa.types.is_decimal256(field.type)
        else table[field.name]
    )
    for field in table.schema
})

df = pl.from_arrow(table_preview)

with pl.Config(
    tbl_cols=-1,
    tbl_width_chars=240,
    fmt_str_lengths=40,
):
    display(df)

event_timestamp_ns,system_timestamp_ns,exchange,instrument_id,symbol,stream_id,side,price,quantity_base,quantity_quote,quantity_contracts,order_count
i64,i64,str,i64,str,str,str,f64,f64,f64,f64,u64
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",0.1,375.9051,37.59051,3.759051e6,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",0.2,13.4226,2.68452,134226.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",0.3,16.1094,4.83282,161094.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",0.4,1.0697,0.42788,10697.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",0.5,413.8663,206.93315,4.138663e6,null
…,…,…,…,…,…,…,…,…,…,…,…
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",1.8,0.1134,0.20412,1134.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",2.0,2.4159,4.8318,24159.0,null
1788220800000000000,null,"""gateio""",17397,"""BTC_USDT""","""gateio:perpetual:linear:BTC_USDT:l2""","""bid""",2.2,0.1364,0.30008,1364.0,null


In [27]:
import json
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import polars as pl

from IPython.display import display

# Find Bybit canonical depth dataset
dataset_id = 89

matches = []

for path in (ROOT / "data").rglob("manifest.json"):
    if "merged" in path.parts:
        continue

    try:
        manifest = json.loads(path.read_text())
    except (OSError, json.JSONDecodeError):
        continue

    if manifest.get("dataset_id") == dataset_id:
        matches.append(path.parent)

assert len(matches) == 1, f"Expected one dataset, found: {matches}"

depth_path = matches[0]

files = sorted(
    depth_path.glob("tasks/task-*/depth/part-*.parquet")
)

assert files, "No depth Parquet files found"

print("DATASET:", dataset_id)
print("PATH:", depth_path)
print("FILES:", len(files))

pf = pq.ParquetFile(files[0])

print("\nARROW SCHEMA:")
print(pf.schema_arrow)

# Read only first 20 rows
batch = next(pf.iter_batches(batch_size=20))
table = pa.Table.from_batches([batch])

# Decimal256 → Float64 for visualization only
preview = pa.table({
    field.name: (
        pc.cast(table[field.name], pa.float64())
        if pa.types.is_decimal256(field.type)
        else table[field.name]
    )
    for field in table.schema
})

df = pl.from_arrow(preview)

with pl.Config(
    tbl_cols=-1,
    tbl_width_chars=240,
    fmt_str_lengths=45,
):
    print("\nFIRST 20 BYBIT DEPTH ROWS:")
    print(df)

DATASET: 89
PATH: /home/pasha/projects/bybit-history-downloader/data/processed/89
FILES: 62

ARROW SCHEMA:
event_timestamp_ns: int64 not null
system_timestamp_ns: int64
exchange: string not null
instrument_id: int64 not null
symbol: string not null
stream_id: string not null
side: string not null
price: decimal256(76, 28) not null
quantity_base: decimal256(76, 28)
quantity_quote: decimal256(76, 28)
quantity_contracts: decimal256(76, 28)
order_count: uint64

FIRST 20 BYBIT DEPTH ROWS:
shape: (20, 12)
┌─────────────────────┬─────────────────────┬──────────┬───────────────┬─────────┬───────────────────────────────────┬──────┬─────────┬───────────────┬────────────────┬────────────────────┬─────────────┐
│ event_timestamp_ns  ┆ system_timestamp_ns ┆ exchange ┆ instrument_id ┆ symbol  ┆ stream_id                         ┆ side ┆ price   ┆ quantity_base ┆ quantity_quote ┆ quantity_contracts ┆ order_count │
│ ---                 ┆ ---                 ┆ ---      ┆ ---           ┆ ---     ┆ --- 

In [28]:
from pathlib import Path
import json
import pyarrow.parquet as pq
import polars as pl
from IPython.display import display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "engine/Cargo.toml").exists()
)

MERGED = ROOT / "data/merged/920003"

manifest = json.loads((MERGED / "manifest.json").read_text())

files = sorted(MERGED.glob("timeline/**/*.parquet"))

print("=" * 80)
print("MARKETFORGE — COMBINED LOGICAL MERGE")
print("=" * 80)

print("\nDATASET:", MERGED)
print("PARQUET FILES:", len(files))
print("TOTAL EVENTS:", f"{manifest['events_written']:,}")

print("\nMANIFEST:")
print(json.dumps(manifest, indent=2, default=str))

print("\nARROW SCHEMA:")
print(pq.ParquetFile(files[0]).schema_arrow)

MARKETFORGE — COMBINED LOGICAL MERGE

DATASET: /home/pasha/projects/bybit-history-downloader/data/merged/920003
PARQUET FILES: 64
TOTAL EVENTS: 455,688,143

MANIFEST:
{
  "protocol_version": 2,
  "timeline_schema_version": 1,
  "job_id": 920003,
  "mode": "combined",
  "input_datasets": [
    {
      "dataset_id": 88,
      "dataset_path": "/home/pasha/projects/bybit-history-downloader/data/.work/full-trade-test/datasets/bybit-perpetual-linear-BTCUSDT-trade-20260901-20260904-d88",
      "manifest_version": 2
    },
    {
      "dataset_id": 90,
      "dataset_path": "/home/pasha/projects/bybit-history-downloader/data/.work/full-trade-test/datasets/binance-perpetual-linear-BTCUSDT-trade-20260901-20260904-d90",
      "manifest_version": 2
    },
    {
      "dataset_id": 91,
      "dataset_path": "/home/pasha/projects/bybit-history-downloader/data/.work/full-trade-test/datasets/okx-perpetual-linear-BTC-USDT-SWAP-trade-20260901-20260904-d91",
      "manifest_version": 2
    },
    {
     

In [29]:
import pyarrow as pa

pf = pq.ParquetFile(files[0])

batch = next(pf.iter_batches(batch_size=100))
table = pa.Table.from_batches([batch])

df = pl.from_arrow(table)

df = df.with_columns(
    pl.from_epoch(
        "event_timestamp_ns",
        time_unit="ns",
    ).alias("datetime_utc"),

    pl.when(pl.col("event_kind") == 0)
    .then(pl.lit("TRADE"))
    .otherwise(pl.lit("DEPTH"))
    .alias("event_type"),
)

with pl.Config(
    tbl_cols=-1,
    tbl_width_chars=240,
    fmt_str_lengths=40,
):
    print(df)

shape: (100, 12)
┌─────────────────────┬─────────────┬────────────┬────────────┬─────────┬───────────────┬──────────────────────┬─────────────────────┬────────────────┬──────────────────┬─────────────────────┬────────────┐
│ event_timestamp_ns  ┆ stream_rank ┆ event_kind ┆ dataset_id ┆ task_id ┆ event_ordinal ┆ canonical_row_offset ┆ canonical_row_count ┆ event_boundary ┆ source_operation ┆ datetime_utc        ┆ event_type │
│ ---                 ┆ ---         ┆ ---        ┆ ---        ┆ ---     ┆ ---           ┆ ---                  ┆ ---                 ┆ ---            ┆ ---              ┆ ---                 ┆ ---        │
│ i64                 ┆ u32         ┆ u8         ┆ u64        ┆ u64     ┆ u64           ┆ u64                  ┆ u64                 ┆ u8             ┆ u8               ┆ datetime[ns]        ┆ str        │
╞═════════════════════╪═════════════╪════════════╪════════════╪═════════╪═══════════════╪══════════════════════╪═════════════════════╪════════════════╪════════

In [30]:
depth_refs = (
    pl.scan_parquet(str(MERGED / "timeline/**/*.parquet"))
    .filter(pl.col("event_kind") == 1)
    .select(
        "event_timestamp_ns",
        "dataset_id",
        "task_id",
        "event_ordinal",
        "canonical_row_offset",
        "canonical_row_count",
        "event_boundary",
        "source_operation",
    )
    .head(100_000)
    .collect()
)

print("DEPTH EVENTS SAMPLED:", len(depth_refs))

print("\nCANONICAL ROW COUNT DISTRIBUTION:")

print(
    depth_refs.select(
        pl.col("canonical_row_count").min().alias("min"),
        pl.col("canonical_row_count").median().alias("median"),
        pl.col("canonical_row_count").mean().alias("mean"),
        pl.col("canonical_row_count").quantile(0.95).alias("p95"),
        pl.col("canonical_row_count").quantile(0.99).alias("p99"),
        pl.col("canonical_row_count").max().alias("max"),
    )
)

print("\nEVENTS BY ROW COUNT:")

print(
    depth_refs.select(
        (pl.col("canonical_row_count") == 0).sum().alias("zero"),
        (pl.col("canonical_row_count") == 1).sum().alias("one"),
        (pl.col("canonical_row_count") > 1).sum().alias("multiple"),
        (pl.col("canonical_row_count") > 100).sum().alias("over_100"),
        (pl.col("canonical_row_count") > 1000).sum().alias("over_1000"),
    )
)

print("\nLARGEST DEPTH EVENTS:")

with pl.Config(tbl_cols=-1, tbl_width_chars=220):
    print(
        depth_refs.sort(
            "canonical_row_count",
            descending=True,
        ).head(15)
    )

DEPTH EVENTS SAMPLED: 100000

CANONICAL ROW COUNT DISTRIBUTION:
shape: (1, 6)
┌─────┬────────┬────────┬─────┬──────┬───────┐
│ min ┆ median ┆ mean   ┆ p95 ┆ p99  ┆ max   │
│ --- ┆ ---    ┆ ---    ┆ --- ┆ ---  ┆ ---   │
│ u64 ┆ f64    ┆ f64    ┆ f64 ┆ f64  ┆ u64   │
╞═════╪════════╪════════╪═════╪══════╪═══════╡
│ 1   ┆ 1.0    ┆ 3.2286 ┆ 1.0 ┆ 27.0 ┆ 53923 │
└─────┴────────┴────────┴─────┴──────┴───────┘

EVENTS BY ROW COUNT:
shape: (1, 5)
┌──────┬───────┬──────────┬──────────┬───────────┐
│ zero ┆ one   ┆ multiple ┆ over_100 ┆ over_1000 │
│ ---  ┆ ---   ┆ ---      ┆ ---      ┆ ---       │
│ u32  ┆ u32   ┆ u32      ┆ u32      ┆ u32       │
╞══════╪═══════╪══════════╪══════════╪═══════════╡
│ 0    ┆ 96896 ┆ 3104     ┆ 176      ┆ 6         │
└──────┴───────┴──────────┴──────────┴───────────┘

LARGEST DEPTH EVENTS:
shape: (15, 8)
┌─────────────────────┬────────────┬─────────┬───────────────┬──────────────────────┬─────────────────────┬────────────────┬──────────────────┐
│ event_timestamp_

In [31]:
print("EVENT BOUNDARIES:")

print(
    depth_refs.group_by("event_boundary")
    .agg(
        pl.len().alias("events"),
        pl.col("canonical_row_count").sum().alias("canonical_levels"),
    )
    .sort("event_boundary")
)

print("\nSOURCE OPERATIONS:")

print(
    depth_refs.group_by("source_operation")
    .agg(
        pl.len().alias("events"),
        pl.col("canonical_row_count").sum().alias("canonical_levels"),
    )
    .sort("source_operation")
)

print("\nBOUNDARY × OPERATION:")

print(
    depth_refs.group_by(
        "event_boundary",
        "source_operation",
    )
    .agg(
        pl.len().alias("events"),
        pl.col("canonical_row_count").sum().alias("canonical_levels"),
    )
    .sort("event_boundary", "source_operation")
)

EVENT BOUNDARIES:
shape: (2, 3)
┌────────────────┬────────┬──────────────────┐
│ event_boundary ┆ events ┆ canonical_levels │
│ ---            ┆ ---    ┆ ---              │
│ u8             ┆ u32    ┆ u64              │
╞════════════════╪════════╪══════════════════╡
│ 0              ┆ 99988  ┆ 190961           │
│ 1              ┆ 12     ┆ 131899           │
└────────────────┴────────┴──────────────────┘

SOURCE OPERATIONS:
shape: (3, 3)
┌──────────────────┬────────┬──────────────────┐
│ source_operation ┆ events ┆ canonical_levels │
│ ---              ┆ ---    ┆ ---              │
│ u8               ┆ u32    ┆ u64              │
╞══════════════════╪════════╪══════════════════╡
│ 0                ┆ 24     ┆ 136413           │
│ 1                ┆ 3130   ┆ 89601            │
│ 2                ┆ 96846  ┆ 96846            │
└──────────────────┴────────┴──────────────────┘

BOUNDARY × OPERATION:
shape: (4, 4)
┌────────────────┬──────────────────┬────────┬──────────────────┐
│ event_bounda

In [32]:
import polars as pl

COMBINED = ROOT / "data/merged/920003"

combined = pl.scan_parquet(
    str(COMBINED / "timeline/**/*.parquet")
)

print("COMBINED DATASET — EVENT DISTRIBUTION")

display(
    combined
    .group_by("event_kind")
    .agg(pl.len().alias("events"))
    .sort("event_kind")
    .collect()
)

print("\nSTREAM DISTRIBUTION")

display(
    combined
    .group_by("stream_rank", "event_kind")
    .agg(pl.len().alias("events"))
    .sort("stream_rank", "event_kind")
    .collect()
)

COMBINED DATASET — EVENT DISTRIBUTION


event_kind,events
u8,u32
0,50106717
1,405581426



STREAM DISTRIBUTION


stream_rank,event_kind,events
u32,u8,u32
0,0,6593458
1,0,11920180
2,0,7680704
3,0,3037806
4,0,3161178
…,…,…
31,1,64869304
32,1,1333233
33,1,11446


In [33]:
import polars as pl

COMBINED = ROOT / "data/merged/920003"

combined = pl.scan_parquet(
    str(COMBINED / "timeline/**/*.parquet")
)

print("COMBINED DATASET — EVENT DISTRIBUTION")

display(
    combined
    .group_by("event_kind")
    .agg(pl.len().alias("events"))
    .sort("event_kind")
    .collect()
)

print("\nSTREAM DISTRIBUTION")

display(
    combined
    .group_by("stream_rank", "event_kind")
    .agg(pl.len().alias("events"))
    .sort("stream_rank", "event_kind")
    .collect()
)

COMBINED DATASET — EVENT DISTRIBUTION


event_kind,events
u8,u32
0,50106717
1,405581426



STREAM DISTRIBUTION


stream_rank,event_kind,events
u32,u8,u32
0,0,6593458
1,0,11920180
2,0,7680704
3,0,3037806
4,0,3161178
…,…,…
31,1,64869304
32,1,1333233
33,1,11446


In [34]:
import pyarrow.parquet as pq
import pyarrow as pa

files = sorted(
    COMBINED.glob("timeline/**/*.parquet")
)

# Select a partition near the middle of the dataset
file = files[len(files) // 2]

print("FILE:", file.relative_to(COMBINED))

pf = pq.ParquetFile(file)

batch = next(pf.iter_batches(batch_size=1000))
table = pa.Table.from_batches([batch])

df = pl.from_arrow(table)

df = df.with_columns(
    pl.from_epoch(
        "event_timestamp_ns",
        time_unit="ns",
    ).alias("datetime_utc"),

    pl.when(pl.col("event_kind") == 0)
    .then(pl.lit("TRADE"))
    .otherwise(pl.lit("DEPTH"))
    .alias("event_type"),
)

with pl.Config(
    tbl_cols=-1,
    tbl_width_chars=240,
    fmt_str_lengths=40,
):
    print(df.head(50))

FILE: timeline/partition-000032/part-000000.parquet
shape: (50, 12)
┌─────────────────────┬─────────────┬────────────┬────────────┬─────────┬───────────────┬──────────────────────┬─────────────────────┬────────────────┬──────────────────┬─────────────────────┬────────────┐
│ event_timestamp_ns  ┆ stream_rank ┆ event_kind ┆ dataset_id ┆ task_id ┆ event_ordinal ┆ canonical_row_offset ┆ canonical_row_count ┆ event_boundary ┆ source_operation ┆ datetime_utc        ┆ event_type │
│ ---                 ┆ ---         ┆ ---        ┆ ---        ┆ ---     ┆ ---           ┆ ---                  ┆ ---                 ┆ ---            ┆ ---              ┆ ---                 ┆ ---        │
│ i64                 ┆ u32         ┆ u8         ┆ u64        ┆ u64     ┆ u64           ┆ u64                  ┆ u64                 ┆ u8             ┆ u8               ┆ datetime[ns]        ┆ str        │
╞═════════════════════╪═════════════╪════════════╪════════════╪═════════╪═══════════════╪═══════════════════

In [35]:
transitions = (
    df
    .with_columns(
        pl.col("event_type")
        .shift(1)
        .alias("previous_type")
    )
    .filter(
        pl.col("event_type") != pl.col("previous_type")
    )
    .select(
        "datetime_utc",
        "event_type",
        "stream_rank",
        "dataset_id",
        "task_id",
        "event_ordinal",
        "canonical_row_count",
    )
)

print("TRADE / DEPTH TRANSITIONS")

with pl.Config(
    tbl_cols=-1,
    tbl_width_chars=220,
):
    print(transitions.head(30))

TRADE / DEPTH TRANSITIONS
shape: (30, 7)
┌────────────────────────────┬────────────┬─────────────┬────────────┬─────────┬───────────────┬─────────────────────┐
│ datetime_utc               ┆ event_type ┆ stream_rank ┆ dataset_id ┆ task_id ┆ event_ordinal ┆ canonical_row_count │
│ ---                        ┆ ---        ┆ ---         ┆ ---        ┆ ---     ┆ ---           ┆ ---                 │
│ datetime[ns]               ┆ str        ┆ u32         ┆ u64        ┆ u64     ┆ u64           ┆ u64                 │
╞════════════════════════════╪════════════╪═════════════╪════════════╪═════════╪═══════════════╪═════════════════════╡
│ 2026-09-02 12:00:00        ┆ DEPTH      ┆ 24          ┆ 96         ┆ 35      ┆ 4839148       ┆ 1                   │
│ 2026-09-02 12:00:00.012700 ┆ TRADE      ┆ 0           ┆ 88         ┆ 1       ┆ 875894        ┆ 1                   │
│ 2026-09-02 12:00:00.027    ┆ DEPTH      ┆ 21          ┆ 89         ┆ 1       ┆ 431918        ┆ 51                  │
│ 2026-